# **Tutorial:** How do modern language models work?

- **Transformers**: architecture based on self-attention. Models learn to predict the next word using large amounts of text. 
- **Pre-training and fine-tuning**: first pre-trained to predict tokens, then adapted (fine-tune / instruct-tune) for conversational or task-specific use.
- **Mixture of Experts (MoE)**: some models (e.g., *Mixtral*) activate only part of the network for each token, allowing high capacity with lower compute cost.
- **Instruct-tuning & DPO**: methods to align models with human instructions (better for chat/help).
- **Embeddings**: models that convert text into vectors (E5, etc.), useful for search and RAG (Retrieval-Augmented Generation).
- **Practical considerations**: model size, GPU memory, latency, safety and moderation. For teaching, we prefer *lighter* models (7B) or *instruct-tuned* checkpoints to avoid extra fine-tuning work.

Below you will find **ready-to-use examples for mini-projects** with open-access models (Mistral, Mixtral, E5, and other recommendations).

In [ ]:
# Installation (run this cell once in your environment)
# Note: Skip this if you're on an environment like Colab/Kaggle that already has these packages.
%pip install transformers 
%pip install torch
%pip install tf-keras
%pip install accelerate 
%pip install huggingface_hub 
%pip install sentence-transformers 
%pip install faiss-cpu 
%pip install datasets

## Example: generation with an instruct model (Mistral)

In [3]:
# Example: Text generation with Mistral-7B-Instruct (Hugging Face Transformers)
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.3"  # instruct-tuned checkpoint
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', torch_dtype=None)
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=256)

prompt = """Act as an educational assistant and briefly summarize what a Transformer is (3–4 lines)."""
res = generator(prompt, do_sample=True, temperature=0.2, top_p=0.95, num_return_sequences=1)
print(res[0]['generated_text'])

2025-09-15 16:02:55.947131: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading tokenizer...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3.
401 Client Error. (Request ID: Root=1-68c87f29-69bbfed72a6c20b0061425c8;658555a5-f6d0-4fc7-b204-8fea54afd7af)

Cannot access gated repo for url https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/main/config.json.
Access to model mistralai/Mistral-7B-Instruct-v0.3 is restricted. You must have access to it and be authenticated to access it. Please log in.

## Example: embeddings (E5-Mistral)

In [5]:

# Example: generating embeddings with intfloat/e5-mistral-7b-instruct
from sentence_transformers import SentenceTransformer
model_name = "intfloat/e5-mistral-7b-instruct"
print("Loading embedding model...")
embedder = SentenceTransformer(model_name)
sentences = ["Artificial intelligence in education", "Language models and embeddings", "Information retrieval with FAISS"]
embs = embedder.encode(sentences, show_progress_bar=False)
print('Dimensions:', len(embs[0]))
print('First embedding (truncated vector):', embs[0][:8])


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.28G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

## Mini-project: RAG with FAISS (small example)

In [ ]:

# Mini-project: Simple RAG (Retrieval-Augmented Generation)
# 1) Create embeddings for documents, 2) index with FAISS, 3) retrieve and use an LLM to answer based on context.
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

docs = [
    "The university has its main campus in Guayaquil.",
    "The enrollment process starts in January and ends in February.",
    "Thesis submissions must follow IEEE format and be at most 25 pages."
]

embed_model = SentenceTransformer("intfloat/e5-mistral-7b-instruct")
doc_embs = embed_model.encode(docs, convert_to_numpy=True)
dim = doc_embs.shape[1]

# Create FAISS index (CPU)
index = faiss.IndexFlatL2(dim)
index.add(doc_embs)
print('Documents indexed:', index.ntotal)

# Query
query = "When does enrollment begin?"
q_emb = embed_model.encode([query], convert_to_numpy=True)
D, I = index.search(q_emb, k=2)
print('Retrieved indices:', I[0])
for i in I[0]:
    print('-', docs[i])

# Use LLM to answer with context
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', torch_dtype=None)
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=150)

context = '\n'.join([docs[i] for i in I[0]])
prompt = f"""Use the following context to answer the question.
Context:\n{context}\nQuestion: {query}\nAnswer:"""
res = generator(prompt, do_sample=False, temperature=0.0, num_return_sequences=1)
print(res[0]['generated_text'])



## Mini-project ideas (each suitable for 1–2 sessions)
- Student support chatbot (RAG + Mistral 7B Instruct).
- Q&A system for university regulations (extract PDF → embeddings → FAISS).
- Automatic summarizer for research papers (input: PDF, output: summary and key points).
- Intent classifier (small dataset, either fine-tuned or embeddings + kNN).
- Simple automation agent (e.g., with n8n + a lightweight LLM, with prompt injection safeguards).



## References and recommended models (state of the art summary)
- Mistral AI: Mistral-7B-Instruct, Mixtral-8x7B (available on Hugging Face). 
- Embeddings: intfloat/e5-mistral-7b-instruct.
- Readings: Hugging Face blog posts on Mixtral and model selection guides.

(Additional updated links can be included outside the notebook.)



# Introducción: ¿Cómo funcionan los modelos de lenguaje actuales?

Breve resumen teórico (para una clase práctica):
- **Transformers**: arquitectura basada en atención (self-attention). Los modelos aprenden a predecir la siguiente palabra usando grandes cantidades de texto. 
- **Pre-entrenamiento y ajuste fino**: primero se pre-entrena el modelo para predecir tokens, luego se adapta (fine-tune / instruct-tune) para tareas conversacionales o específicas.
- **Modelos de mezcla de expertos (Mixture of Experts, MoE)**: algunos modelos (ej. *Mixtral*) activan sólo partes del modelo para cada token, permitiendo mayor capacidad con menor coste de cómputo efectivo.
- **Instruct-tuning & DPO**: técnicas para alinear modelos con instrucciones humanas (mejor comportamiento en chat/ayuda).
- **Embeddings**: modelos que convierten texto a vectores (E5, etc.) útiles para búsqueda y RAG (Retrieval-Augmented Generation).
- **Cuestiones prácticas**: tamaño del modelo, memoria GPU, latencia, seguridad y moderación. Para clases, preferimos modelos *ligeros* (7B) o *instruct* ya afinados para evitar trabajo extra de fine-tuning.

A continuación verás ejemplos prácticos **listos para mini-proyectos** usando modelos de libre acceso (Mistral, Mixtral, E5, y recomendaciones actuales).

> **Nota**: Los nombres de modelos y recomendaciones se basan en el estado del arte público (Hugging Face, blogs, anuncios). Referencias en la última celda.


In [ ]:

# Instalación (ejecutar en una celda de notebook una vez)
# Nota: evita ejecutar instalaciones si estás en un entorno ya preparado (colab, kaggle, etc.).
!pip install --upgrade pip
!pip install -q transformers accelerate huggingface_hub sentence-transformers faiss-cpu datasets


## Ejemplo: generación con un modelo instruct (Mistral)

In [ ]:

# Ejemplo: generación con Mistral-7B-Instruct (Hugging Face Transformers)
# Este ejemplo usa la API de transformers. Dependiendo de tu entorno necesitarás GPU y suficiente RAM.
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.3"  # instruct-tuned checkpoint
print("Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
print("Cargando modelo (sin bajar pesos si se usa HuggingFace Inference)...")
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', torch_dtype=None)
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=256)

prompt = """Actúa como un asistente educativo y resume brevemente qué es un transformador (3-4 líneas)."""
res = generator(prompt, do_sample=True, temperature=0.2, top_p=0.95, num_return_sequences=1)
print(res[0]['generated_text'])


## Ejemplo: embeddings (E5-Mistral)

In [ ]:

# Ejemplo: generar embeddings con intfloat/e5-mistral-7b-instruct
from sentence_transformers import SentenceTransformer
model_name = "intfloat/e5-mistral-7b-instruct"
print("Cargando modelo de embeddings...")
embedder = SentenceTransformer(model_name)
sentences = ["La inteligencia artificial en la educación", "Modelos de lenguaje y embeddings", "Recuperación de información con FAISS"]
embs = embedder.encode(sentences, show_progress_bar=False)
print('Dimensiones:', len(embs[0]))
print('Primer embedding (vector truncated):', embs[0][:8])


## Mini-proyecto: RAG con FAISS (pequeño ejemplo)

In [ ]:

# Mini-proyecto: RAG (Retrieval-Augmented Generation) simple
# 1) Crear embeddings de documentos, 2) indexar con FAISS, 3) recuperar y usar un LLM para generar respuesta basada en contexto.
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

docs = [
    "La Universidad tiene un campus principal en Guayaquil.",
    "El proceso de matrícula empieza en enero y termina en febrero.",
    "Para tesis, se requiere formato IEEE y máximo 25 páginas."
]

embed_model = SentenceTransformer("intfloat/e5-mistral-7b-instruct")
doc_embs = embed_model.encode(docs, convert_to_numpy=True)
dim = doc_embs.shape[1]

# Crear índice FAISS (CPU)
index = faiss.IndexFlatL2(dim)
index.add(doc_embs)
print('Documentos indexados:', index.ntotal)

# Query
query = "¿Cuándo comienza la matrícula?"
q_emb = embed_model.encode([query], convert_to_numpy=True)
D, I = index.search(q_emb, k=2)
print('Índices recuperados:', I[0])
for i in I[0]:
    print('-', docs[i])

# Ahora usar LLM para generar respuesta con el contexto
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', torch_dtype=None)
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=150)

context = '\n'.join([docs[i] for i in I[0]])
prompt = f"""Usa la información de contexto para responder a la pregunta.
Contexto:\n{context}\nPregunta: {query}\nRespuesta:"""
res = generator(prompt, do_sample=False, temperature=0.0, num_return_sequences=1)
print(res[0]['generated_text'])



## Ideas de mini-proyectos (cada uno para 1-2 sesiones)
- Chatbot de atención estudiantil (uso de RAG + Mistral 7B Instruct).
- Motor de preguntas y respuestas sobre normativa de la universidad (extraer PDF -> embeddings -> FAISS).
- Generador de resúmenes automáticos para artículos científicos (entrada: PDF, salida: resumen y bullets).
- Clasificador de intenciones (pequeño dataset, fine-tune o usar embeddings + kNN).
- Agente simple con n8n + un LLM ligero para automatizar respuestas por correo (recuerda medidas de seguridad contra prompt injections).



## Referencias y modelos recomendados (estado del arte resumido)
- Mistral AI: Mistral-7B-Instruct, Mixtral-8x7B (Hugging Face repos). 
- Embeddings: intfloat/e5-mistral-7b-instruct.
- Lecturas: Hugging Face blog sobre Mixtral y guías de selección de modelos.

(En la versión entregada por chat incluyo enlaces y fuentes externas actualizadas).


Autor:  
Manuel Eugenio Morocho Cayamcela, PhD

# **Introducción a Large Language Models (LLMs) con Python**

## Objetivo:
El objetivo de este taller es introducir a los estudiantes a los conceptos básicos de los large language models (LLMs) y proporcionarles una experiencia práctica utilizando bibliotecas de Hugging Face para trabajar con modelos preentrenados de lenguaje natural.

## Contenido:
1. Introducción a los Large Language Models (LLMs)
2. Configuración del entorno
3. Ejemplos prácticos
4. Ejercicio

## 1. Introducción a los Large Language Models (LLMs)

Los large language models (LLMs) son modelos de inteligencia artificial que han sido entrenados en grandes cantidades de texto para comprender y generar lenguaje natural. Estos modelos pueden realizar una variedad de tareas de procesamiento de lenguaje natural (NLP) como traducción, resumen, generación de texto, respuesta a preguntas, entre otros.

## 2. Configuración del entorno

Para este taller, utilizaremos la biblioteca `transformers` de Hugging Face. Primero, necesitamos instalar las bibliotecas necesarias.

In [1]:
# Instalación de la biblioteca transformers y torch
%pip install transformers
%pip install torch
%pip install tf-keras

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Importamos `pipeline` de la librería `transformers`

In [2]:
from transformers import pipeline

2025-09-02 20:47:42.615588: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 3. Ejemplos prácticos de carga y uso de modelos preentrenados

### 3.1 Generación de texto (`text-generation`)

Vamos a cargar un modelo preentrenado de Hugging Face y usarlo para generar texto.

In [4]:
# Cargamos un pipeline de generación de texto
generator = pipeline('text-generation', model='gpt2')

# Generamos texto con el modelo cargado
text = generator(
    #"Había una vez, un niño que tenía una bicicleta",
    "Once upon a time there was a kid who had a bike", 
    max_length=50, # Longitud máxima del texto generado.
    num_return_sequences=3,
    temperature=0.7, # Este parámetro controla la aleatoriedad de las respuestas. Valores más bajos hacen que el modelo sea más predecible, mientras que valores más altos lo hacen más creativo.
    )

#print(text[0]['generated_text'])

# Imprimimos las secuencias generadas
for i, sequence in enumerate(text):
    print(f"Secuencia {i+1}: {sequence['generated_text']}\n")

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Secuencia 1: Once upon a time there was a kid who had a bike and he was biking with his mom, so he would say, "What do you want?"

"I'm not going to change my mind," he said. "I mean,

Secuencia 2: Once upon a time there was a kid who had a bike. He was on his way to school. One of the kids was taking a photo to see if I was being nice. I'd never met him before, but he had been on the

Secuencia 3: Once upon a time there was a kid who had a bike rack and he could ride it all day. He was so big that he could ride it all day. Just about every kid in school was going to have one, and he'd go on



Ahora usamos el modelo para la generación de texto mediante prompts

In [5]:
# Ejemplo de generación de texto con diferentes prompts
prompts = [
    "In the future, AI will",
    "The secret to happiness is",
    "The quick brown fox"
]

for prompt in prompts:
    text = generator(prompt, max_length=50, num_return_sequences=1)
    print(f"Prompt: {prompt}")
    print(f"Generated Text: {text[0]['generated_text']}\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: In the future, AI will
Generated Text: In the future, AI will replace those technologies as a central driving force for social change worldwide. As they go, AI will not only change how robots do things, but what kinds of machines they form and how much they work with each other.




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: The secret to happiness is
Generated Text: The secret to happiness is getting up in the morning, in the morning.

1. Go back to your life, not just the morning.

2. Go to your daily routine, not just the past time.

3.

Prompt: The quick brown fox
Generated Text: The quick brown fox is quite an animal and it will need some help at some stage.

Step 2. Remove the skin from your fur coat, to a wet, clean looking color.

Step 3. Clean and dry your fur and



### 3.2 Respuesta a preguntas (`question-answering`)

Podemos usar un pipeline de respuesta a preguntas para encontrar respuestas dentro de un contexto proporcionado.

In [6]:
# Cargamos un pipeline de respuesta a preguntas
qa_pipeline = pipeline('question-answering')

# Definimos el contexto y la pregunta
#context = "You are a mathematics teacher that can explain complex concepts in simple terms."
context = (
    "Artificial intelligence (AI) is a branch of computer science that aims to create machines that can perform tasks that would normally require human intelligence. "
    "Machine learning (ML) is a subset of AI that involves the use of algorithms and statistical models to enable computers to improve their performance on a task through experience. "
    "One common application of ML is in the field of natural language processing (NLP), where algorithms are used to understand and generate human language. "
    "For example, GPT-4o is a state-of-the-art language model developed by OpenAI that can generate human-like text based on a given prompt."
)
question = "What is machine learning?"

# Obtenemos la respuesta
result = qa_pipeline(question=question, context=context)
print(f"Pregunta: {question}")
print(f"Respuesta: {result['answer']}")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


Pregunta: What is machine learning?
Respuesta: a subset of AI that involves the use of algorithms and statistical models


### 3.3 Resumen de texto (`summarization`)

Usa un modelo preentrenado para generar un resumen de un texto largo.

In [7]:
# Cargamos un pipeline de resumen
summarizer = pipeline('summarization')

# Texto largo para resumir
long_text = (
"La inteligencia artificial (IA) se refiere a la simulación de la inteligencia humana en máquinas que están programadas para pensar como humanos e imitar sus acciones. El término también se puede aplicar a cualquier máquina que exhiba rasgos asociados con una mente humana como el aprendizaje y la resolución de problemas. La característica ideal de la inteligencia artificial es su capacidad para racionalizar y tomar acciones que tengan la mejor posibilidad de alcanzar un objetivo específico. Un subconjunto de la inteligencia artificial es el aprendizaje automático, que se refiere a la idea de que los sistemas informáticos pueden aprender de datos, identificar patrones y tomar decisiones con una mínima intervención humana."
)

# Generamos el resumen
summary = summarizer(long_text, max_length=50, min_length=25)
print("Resumen:")
print(summary[0]['summary_text'])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


Resumen:
 La inteligencia artificial (IA) se refiere a la simulación of la inteligenia humana . La característica ideal de la IAI is su capacidad for racionalizar y tomar


### 3.4 Traducción de texto (`translation`)

Usamos un modelo traductor preentrenado de inglés a español

In [8]:
%pip install sentencepiece
import sentencepiece

# Cargamos un pipeline de traducción especificando el modelo
translation_pipeline = pipeline('translation', model='Helsinki-NLP/opus-mt-en-es')

# Definimos el texto a traducir
text = "Persistent homology is a method for computing topological features of a space at different spatial resolutions. More persistent features are detected over a wide range of spatial scales and are deemed more likely to represent true features of the underlying space rather than artifacts of sampling, noise, or particular choice of parameters"

# Obtenemos la traducción.
result = translation_pipeline(text)
print(f"Texto original: {text}")
print(f"Traducción: {result[0]['translation_text']}")

Note: you may need to restart the kernel to use updated packages.


/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Texto original: Persistent homology is a method for computing topological features of a space at different spatial resolutions. More persistent features are detected over a wide range of spatial scales and are deemed more likely to represent true features of the underlying space rather than artifacts of sampling, noise, or particular choice of parameters
Traducción: La homología persistente es un método para calcular las características topológicas de un espacio en diferentes resoluciones espaciales. Se detectan características más persistentes en una amplia gama de escalas espaciales y se considera más probable que representen características verdaderas del espacio subyacente en lugar de artefactos de muestreo, ruido o elección particular de parámetros.


### 3.4 Análisis de sentimientos (`sentiment-analysis`)

Ahora usaremos un modelo de análisis de sentimientos para predecir si el texto es positivo o negativo.

In [12]:
# Load the sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Text to analyze sentiment
text = "The price of the NVIDIA stock will increase tomorrow."
#text = "Today I have a quiz for the course Artificial Intelligence."
#text = "A cat is running in green fields in the summer."

# Perform sentiment analysis
sentiment_result = sentiment_analyzer(text)

# Print the sentiment result
print("Sentiment Analysis Result:")
for result in sentiment_result:
    print(f"Label: {result['label']}, Score: {result['score']}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Sentiment Analysis Result:
Label: POSITIVE, Score: 0.9976094961166382


### 3.5 Preguntando a una tabla (`table-question-answering`)

Ahora vamos a hacer preguntas a un DataFrame de Pandas

In [ ]:
import pandas as pd

# Load a table-question-answering pipeline
#table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
table_qa = pipeline("table-question-answering")

#Create a table as a pandas DataFrame
table = {
    "Name": ["Karen", "Anna", "Pedro"],
    "Age": ["22", "31", "19"],  # Ensure all entries are strings
    "Interest": ["Cybersecurity", "Numerical Analysis", "Functional Analysis"]
}

# Question about the table
question= "What is the interest of Karen?"
#question = "What is the age of Pedro?"
#question = "What is the average age of the table?" # 22 + 31 + 19 = 72 / 3 = 24

# Generate the answer
answer = table_qa(table=table, query=question)

# Print the question and the answer
print("Question:", question)
print("Answer:", answer['answer'])

No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision 69ceee2 (https://huggingface.co/google/tapas-base-finetuned-wtq).
Using a pipeline without specifying a model name and revision in production is not recommended.


Question: What is the average age of the table?
Answer: AVERAGE > 22, 31, 19


## Otras tareas:

- 'audio-classification'
- 'automatic-speech-recognition'
- 'conversational'
- 'depth-estimation'
- 'document-question-answering'
- 'feature-extraction'
- 'fill-mask'
- 'image-classification'
- 'image-feature-extraction'
- 'image-segmentation'
- 'image-to-image'
- 'image-to-text'
- 'mask-generation'
- 'ner', 'object-detection'
- 'question-answering'
- 'sentiment-analysis'
- 'summarization'
- 'table-question-answering'
- 'text-classification'
- 'text-generation'
- 'text-to-audio'
- 'text-to-speech'
- 'text2text-generation'
- 'token-classification'
- 'translation'
- 'video-classification'
- 'visual-question-answering'
- 'vqa'
- 'zero-shot-audio-classification'
- 'zero-shot-classification'
- 'zero-shot-image-classification'
- 'zero-shot-object-detection'
- 'translation_XX_to_YY'

### **Actividad Individual:** Comparación de Modelos de Hugging Face

En esta actividad, los estudiantes elegirán dos modelos disponibles en Hugging Face para cualquier tipo de tarea diferente a las que se encuentran en este notebook, los probarán con 3 diferentes datos de entrada (dependiendo del tipo de tarea) y realizarán un análisis crítico de los resultados generados por cada modelo.

#### Objetivo

El objetivo de esta actividad es que los estudiantes comparen diferentes modelos de Hugging Face en una tarea específica y desarrollen habilidades críticas para evaluar la calidad de los resultados generados.

#### Instrucciones

1. **Configura el Entorno**:
    - Asegúrate de tener instalado Python y la biblioteca `transformers`. Si no los tienes instalados, puedes hacerlo con los siguientes comandos:
    ```sh
    pip install transformers
    ```

3. **Elige Dos Modelos de Hugging Face**:
    - Visita la [página de modelos de Hugging Face](https://huggingface.co/models) y elige dos modelos para la tarea que deseas realizar.

4. **Prueba los Modelos con el Conjunto de Datos**:
    - Utiliza los modelos seleccionados para realizar la tarea con el conjunto de datos elegido (dependiendo del tipo de tarea). Asegúrate de utilizar el mismo conjunto de datos para ambos modelos para que la comparación sea justa.

5. **Analiza los Resultados**:
    - Revisa los resultados generados por ambos modelos y compáralos. Evalúa la relevancia y coherencia de cada resultado.

6. **Entrega**:

*Tiempo estimado para completar la terea:* 30min

Usa el siguiente espacio para escribir un informe breve que incluya:  

**1. Una descripción de la tarea que realizaste:**

- Escribe aquí..

**2. El conjunto de datos (o prompt) original:**  

- Escribe aquí..

**3. Los resultados generados por ambos modelos:**  

- Escribe aquí..


In [ ]:
# Escribe el código aquí abajo



**4. Un análisis crítico de cada resultado, incluyendo observaciones sobre su relevancia y coherencia.**  

- Escribe aquí..

**5. Graba este Notebook como PDF y súbelo a la actividad designada en Moodle.**  